In [1]:
# import time, random
from urllib.parse import urljoin, urlparse

# from bs4 import BeautifulSoup
# from selenium.webdriver.common.by import By
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC
import re
import json
import time
import random
from datetime import datetime, timezone

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager


BASE_LIST_URL = "https://www.daft.ie/new-homes-for-sale/ireland?sort=publishDateDesc&page={page}"
MAX_PAGE = 20

def extract_new_home_links(listing_html: str) -> list[str]:
    """
    Pull development links from a new-homes results page.
    We target URLs that contain /new-home-for-sale/ (the development detail pages).
    """
    soup = BeautifulSoup(listing_html, "lxml")
    links = set()

    for a in soup.select('a[href*="/new-home-for-sale/"]'):
        href = a.get("href")
        if not href:
            continue
        full = urljoin("https://www.daft.ie", href)

        # normalize: drop query/fragment
        u = urlparse(full)
        full = f"{u.scheme}://{u.netloc}{u.path}"

        links.add(full)

    return sorted(links)

def collect_all_development_urls(driver) -> list[str]:
    all_urls = set()

    for page in range(1, MAX_PAGE + 1):
        url = BASE_LIST_URL.format(page=page)
        driver.get(url)
        time.sleep(random.uniform(1.2, 2.2))
        # wait until some results links appear (best-effort)
        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'a[href*="/new-home-for-sale/"]'))
            )
        except Exception:
            pass  # still try parsing whatever loaded

        # optional: scroll a bit in case lazy-loaded
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(random.uniform(5.0, 9.0))

        page_links = extract_new_home_links(driver.page_source)
        print(f"Page {page}: found {len(page_links)} development links")
        all_urls.update(page_links)

        time.sleep(random.uniform(1.2, 2.2))

    return sorted(all_urls)


In [2]:
import tempfile
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
options.add_argument("--window-size=1400,900")

# IMPORTANT: use a unique profile dir each run to avoid "DevToolsActivePort" + locks
profile_dir = tempfile.mkdtemp(prefix="selenium-daft-")
options.add_argument(f"--user-data-dir={profile_dir}")

# Good stability flags (esp. for DevToolsActivePort crashes)
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--remote-debugging-port=0")



driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# STEP 1: Open Daft and wait for manual cookie acceptance
driver.get("https://www.daft.ie/new-homes-for-sale/ireland?sort=publishDateDesc&page=1")

print("⏳ You have 30 seconds to accept cookies in the browser…")
time.sleep(30)
print("Continuing scrape")

# STEP 2: Now collect all development URLs
development_urls = collect_all_development_urls(driver)

print("Total unique development URLs:", len(development_urls))

⏳ You have 30 seconds to accept cookies in the browser…
Continuing scrape
Page 1: found 55 development links
Page 2: found 52 development links
Page 3: found 62 development links
Page 4: found 57 development links
Page 5: found 55 development links
Page 6: found 61 development links
Page 7: found 62 development links
Page 8: found 61 development links
Page 9: found 62 development links
Page 10: found 55 development links
Page 11: found 62 development links
Page 12: found 49 development links
Page 13: found 58 development links
Page 14: found 54 development links
Page 15: found 55 development links
Page 16: found 59 development links
Page 17: found 57 development links
Page 18: found 62 development links
Page 19: found 55 development links
Page 20: found 28 development links
Total unique development URLs: 1121


In [9]:
from pathlib import Path

OUTPUT_DIR = Path("C:/Users/ciara/myenv/New Homes Ireland/Daft - Dan/Daft-MyHome-Scraper-main/data")

# make sure the directory exists
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from datetime import datetime

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = OUTPUT_DIR / f"daft_listings_newhomes_{ts}.csv"
# output_file = OUTPUT_DIR / f"daft_listings_newhomes.csv"

df = pd.DataFrame(
    development_urls,
    columns=[
        "url"
    ],
)


df.to_csv(output_file, index=False)

print(f"Saved to {output_file}")

Saved to C:\Users\ciara\myenv\New Homes Ireland\Daft - Dan\Daft-MyHome-Scraper-main\data\daft_listings_newhomes_20260512_104622.csv


In [10]:
from pathlib import Path

folder = Path(r"C:/Users/ciara/myenv/New Homes Ireland/Daft - Dan/Daft-MyHome-Scraper-main/data")

# Find matching files
files = list(folder.glob("daft_listings_newhomes_*.csv"))

# Pick latest based on timestamp in filename
latest_file = max(
    files,
    key=lambda f: f.stem.split("_")[-2] + f.stem.split("_")[-1]
)

print(latest_file)

C:\Users\ciara\myenv\New Homes Ireland\Daft - Dan\Daft-MyHome-Scraper-main\data\daft_listings_newhomes_20260512_104622.csv
